In [ ]:
import importlib
import json
import subprocess
import sys
import tomllib
from importlib.metadata import version
from pathlib import Path

if any(
    name == "eurusd_research" or name.startswith("eurusd_research.")
    for name in sys.modules
):
    raise RuntimeError("Restart the kernel, then begin again at Cell 2.")

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / ".git").exists()),
    None,
)

if REPO_ROOT is None:
    REPO_ROOT = Path(
        input("Full path to the existing repository: ").strip().strip('"')
    ).expanduser().resolve()

if not (REPO_ROOT / ".git").exists():
    raise FileNotFoundError("An existing Git checkout is required.")

for package in ("numpy", "pandas", "pyarrow", "duckdb", "pytz", "pytest", "tzdata"):
    print(f"{package:12s} {version(package)}")

project = tomllib.loads(
    (REPO_ROOT / "pyproject.toml").read_text(encoding="utf-8")
)
if "pytz" not in project["project"]["dependencies"]:
    raise RuntimeError(
        "Add pytz to pyproject.toml as described in the Notebook 01 repair "
        "before continuing."
    )

for relative in (
    "src/eurusd_research/features.py",
    "tests/test_features.py",
    "configs/research_config.example.json",
    "configs/research_config.local.json",
):
    print("EXISTS" if (REPO_ROOT / relative).exists() else "MISSING", relative)

new_files = {}

In [ ]:
# ## Sampling and feature definitions

# At clock time t, the selected source snapshot must have timestamp ≤ t.

# The snapshot timestamp is retained even when its prices are stale.
# Usable price features become missing when the snapshot is too old.

# For EURUSD:

#     mid_price = (bid + ask) / 2
#     spread_pips = (ask - bid) / 0.0001

# Displayed-size features:

#     imbalance = (bid_size - ask_size) / (bid_size + ask_size)

#     weighted_quote_price =
#         (ask * bid_size + bid * ask_size) / (bid_size + ask_size)

# The weighted quote price is a candidate predictor, not established
# fair value or a calibrated microprice.

# ## Trailing windows

# The grid contains every second, so shifting by h rows means h seconds.

# - h-second price change: mid(t) minus mid(t-h), in pips.
# - h-second log return: log(mid(t) / mid(t-h)).
# - W-second realized volatility: square root of the sum of W squared
#   one-second pip changes. It is measured in pips and is not annualized.
# - Snapshot count: number of source records in (t-W, t].
#   This measures observed quote activity, not executed trade flow.
# - Spread and size changes compare t with t-h.

# Trailing price calculations require every intervening one-second
# price step to be usable. Gaps and observed invalid price snapshots
# break continuity, including invalid snapshots followed by recovery
# within the same second.

# ## Observable change ages

# Bid and ask price-change ages use comparisons between consecutive
# usable source snapshots. Invalid comparisons and substantial gaps
# reset that knowledge.

# The ages are capped at a configurable observation window. Missing
# values can mean an older change or insufficient knowledge.

# ## Time features

# London and New York local times use timezone-aware conversions.

# Initial session conventions are Monday–Friday, 08:00 inclusive to
# 17:00 exclusive in each city's local time. Overlap is allowed.
# # These are research conventions, not universal market boundaries.

In [ ]:
feature_calculations = r'''\
"""Causal clock-time sampling, features, and bounded feature storage."""

import hashlib
import json
import os
import re
import tempfile
import time
from importlib.metadata import version
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from .data import _sha256, _verify_parts, _writer_lock

QUOTE_COLUMNS = [
    "timestamp_utc", "source_row", "source_file", "source_sha256",
    "bid", "ask", "bid_size", "ask_size", "source_gap_seconds",
    "flag_missing_size", "flag_invalid_size",
    "flag_zero_size", "flag_negative_size",
]


def _utc(value):
    value = pd.Timestamp(value)
    if value.tzinfo is None:
        raise ValueError("Timestamps must include a timezone.")
    value = value.tz_convert("UTC")
    if value.microsecond or value.nanosecond:
        raise ValueError("Processing boundaries must be aligned to whole seconds.")
    return value


def lookback_seconds(settings):
    windows = (
        settings["return_windows_seconds"]
        + settings["volatility_windows_seconds"]
        + settings["activity_windows_seconds"]
        + settings["change_windows_seconds"]
        + [settings["observed_price_change_window_seconds"]]
    )
    if not windows or any(type(value) is not int or value < 1 for value in windows):
        raise ValueError("Feature windows must be positive integer seconds.")
    if not (
        0 <= settings["max_staleness_seconds"]
        <= settings["substantial_gap_seconds"]
    ):
        raise ValueError("Staleness must be nonnegative and no larger than the gap limit.")
    return max(windows) + 1


def normalise_feature_types(frame):
    frame = frame.copy()
    time_columns = {"feature_timestamp_utc", "source_timestamp_utc"}
    string_columns = {"source_file", "source_sha256", "state_reason"}

    for name in frame:
        if name in time_columns:
            frame[name] = pd.to_datetime(frame[name], utc=True).astype(
                "datetime64[us, UTC]"
            )
        elif name in string_columns:
            frame[name] = frame[name].astype("string")
        elif name == "source_row":
            frame[name] = frame[name].astype("Int64")
        elif pd.api.types.is_bool_dtype(frame[name].dtype):
            frame[name] = frame[name].astype(bool)
        else:
            frame[name] = frame[name].astype("float64")
    return frame


def _event_bins(timestamps, grid, mask=None):
    if mask is not None:
        timestamps = timestamps.loc[mask]
    counts = timestamps.dt.ceil("s").value_counts(sort=False)
    return counts.reindex(grid, fill_value=0).to_numpy(dtype="int64")


def build_clock_features(quotes, start, end, settings, coverage_start):
    start, end = _utc(start), _utc(end)
    if start >= end:
        raise ValueError("The processing interval must have positive length.")

    context_start = start - pd.Timedelta(seconds=lookback_seconds(settings))
    grid = pd.date_range(
        context_start, end, freq="1s", inclusive="left"
    ).astype("datetime64[us, UTC]")

    if len(grid) > settings["max_grid_rows_per_block"]:
        raise ValueError("Grid exceeds the configured block limit.")

    q = quotes[QUOTE_COLUMNS].copy()
    q["timestamp_utc"] = pd.to_datetime(q["timestamp_utc"], utc=True).astype(
        "datetime64[us, UTC]"
    )
    if q["timestamp_utc"].isna().any():
        raise ValueError("Canonical input quotes must have usable UTC timestamps.")

    q = q.loc[q["timestamp_utc"].le(grid[-1])].sort_values(
        ["timestamp_utc", "source_row"], kind="stable"
    ).reset_index(drop=True)

    position = q["timestamp_utc"].searchsorted(context_start, side="left")
    q = q.iloc[max(0, position - 1):].reset_index(drop=True)

    usable_raw_price = (
        np.isfinite(q["bid"]) & np.isfinite(q["ask"])
        & q["bid"].gt(0) & q["ask"].gt(0)
        & q["ask"].ge(q["bid"])
    )
    raw_total_size = q["bid_size"] + q["ask_size"]
    usable_raw_size = (
        np.isfinite(q["bid_size"]) & np.isfinite(q["ask_size"])
        & q["bid_size"].ge(0) & q["ask_size"].ge(0)
        & np.isfinite(raw_total_size) & raw_total_size.gt(0)
    )
    raw_gap = q["source_gap_seconds"].gt(settings["substantial_gap_seconds"])
    comparable = (
        usable_raw_price
        & usable_raw_price.shift(fill_value=False)
        & ~raw_gap
    )
    knowledge_group = (~comparable).cumsum()

    for side in ("bid", "ask"):
        changed = comparable & q[side].ne(q[side].shift())
        event_time = q["timestamp_utc"].where(changed)
        q[f"_{side}_change_time"] = event_time.groupby(knowledge_group).ffill()

    right = q.rename(columns={"timestamp_utc": "source_timestamp_utc"})
    frame = pd.DataFrame({"feature_timestamp_utc": grid})
    sampled = pd.merge_asof(
        frame,
        right,
        left_on="feature_timestamp_utc",
        right_on="source_timestamp_utc",
        direction="backward",
        allow_exact_matches=True,
    )

    for name in ("source_timestamp_utc", "source_row", "source_file", "source_sha256"):
        frame[name] = sampled[name]

    age = (
        frame["feature_timestamp_utc"] - frame["source_timestamp_utc"]
    ).dt.total_seconds()
    has_source = frame["source_timestamp_utc"].notna()
    if age.loc[has_source].lt(0).any():
        raise RuntimeError("Future snapshot entered current features.")

    usable_price = (
        np.isfinite(sampled["bid"]) & np.isfinite(sampled["ask"])
        & sampled["bid"].gt(0) & sampled["ask"].gt(0)
        & sampled["ask"].ge(sampled["bid"])
    )
    stale = has_source & age.gt(settings["max_staleness_seconds"])
    price_valid = has_source & ~stale & usable_price

    frame["snapshot_staleness_seconds"] = age
    frame["price_valid"] = price_valid
    frame["state_reason"] = np.select(
        [~has_source, stale, ~usable_price],
        ["no_source_quote", "stale_snapshot", "invalid_latest_price"],
        default="usable_price",
    )

    frame["bid"] = sampled["bid"].where(price_valid)
    frame["ask"] = sampled["ask"].where(price_valid)
    frame["mid_price"] = 0.5 * frame["bid"] + 0.5 * frame["ask"]
    frame["spread_pips"] = (frame["ask"] - frame["bid"]) / settings["pip_size"]
    frame["locked_quote"] = price_valid & sampled["ask"].eq(sampled["bid"])
    frame["wide_spread_observation"] = frame["spread_pips"].gt(
        settings["wide_spread_review_pips"]
    )

    for side in ("bid", "ask"):
        size = sampled[f"{side}_size"]
        frame[f"{side}_size"] = size.where(
            price_valid & np.isfinite(size) & size.ge(0)
        )

    total_size = frame["bid_size"] + frame["ask_size"]
    size_valid = (
        price_valid & np.isfinite(total_size) & total_size.gt(0)
        & frame["bid_size"].notna() & frame["ask_size"].notna()
    )
    denominator = total_size.where(size_valid)
    frame["size_pair_valid"] = size_valid
    frame["size_imbalance"] = (
        frame["bid_size"] - frame["ask_size"]
    ) / denominator
    frame["size_weighted_quote_price"] = (
        frame["ask"] * frame["bid_size"]
        + frame["bid"] * frame["ask_size"]
    ) / denominator
    frame["size_weighted_minus_mid_pips"] = (
        frame["size_weighted_quote_price"] - frame["mid_price"]
    ) / settings["pip_size"]

    for name in (
        "flag_missing_size", "flag_invalid_size",
        "flag_zero_size", "flag_negative_size",
    ):
        frame[f"sampled_{name}"] = sampled[name].eq(True)

    timestamps = q["timestamp_utc"]
    raw_counts = pd.Series(_event_bins(timestamps, grid))
    gap_event = pd.Series(_event_bins(timestamps, grid, raw_gap)).gt(0)
    invalid_price_event = pd.Series(
        _event_bins(timestamps, grid, ~usable_raw_price)
    ).gt(0)
    invalid_size_event = pd.Series(
        _event_bins(timestamps, grid, ~usable_raw_size)
    ).gt(0)

    step_valid = (
        price_valid & price_valid.shift(fill_value=False)
        & ~gap_event & ~invalid_price_event
    )
    size_step_valid = (
        step_valid & size_valid & size_valid.shift(fill_value=False)
        & ~invalid_size_event
    )

    frame["source_gap_observed_1s"] = gap_event
    frame["invalid_price_observed_1s"] = invalid_price_event
    frame["unusable_size_observed_1s"] = invalid_size_event
    frame["price_path_break"] = ~step_valid

    pip_step = (
        frame["mid_price"].diff() / settings["pip_size"]
    ).where(step_valid)

    for horizon in settings["return_windows_seconds"]:
        complete = step_valid.rolling(horizon, min_periods=horizon).sum().eq(horizon)
        frame[f"mid_change_{horizon}s_pips"] = (
            (frame["mid_price"] - frame["mid_price"].shift(horizon))
            / settings["pip_size"]
        ).where(complete)
        frame[f"log_return_{horizon}s"] = np.log(
            frame["mid_price"] / frame["mid_price"].shift(horizon)
        ).where(complete)

    for window in settings["volatility_windows_seconds"]:
        frame[f"realized_vol_{window}s_pips"] = (
            pip_step.pow(2).rolling(window, min_periods=window).sum().pow(0.5)
        )

    coverage_start = pd.Timestamp(coverage_start).tz_convert("UTC")
    for window in settings["activity_windows_seconds"]:
        complete_history = (
            has_source
            & frame["feature_timestamp_utc"].sub(
                pd.Timedelta(seconds=window)
            ).ge(coverage_start)
        )
        frame[f"snapshot_count_{window}s"] = raw_counts.rolling(
            window, min_periods=window
        ).sum().where(complete_history)

    for horizon in settings["change_windows_seconds"]:
        price_complete = step_valid.rolling(
            horizon, min_periods=horizon
        ).sum().eq(horizon)
        size_complete = size_step_valid.rolling(
            horizon, min_periods=horizon
        ).sum().eq(horizon)

        frame[f"spread_change_{horizon}s_pips"] = (
            frame["spread_pips"].diff(horizon).where(price_complete)
        )
        for side in ("bid", "ask"):
            frame[f"{side}_size_change_{horizon}s"] = (
                frame[f"{side}_size"].diff(horizon).where(size_complete)
            )

    age_window = settings["observed_price_change_window_seconds"]
    for side in ("bid", "ask"):
        change_age = (
            frame["feature_timestamp_utc"] - sampled[f"_{side}_change_time"]
        ).dt.total_seconds()
        known_recent = price_valid & change_age.between(0, age_window)
        frame[f"{side}_price_change_age_seconds"] = change_age.where(known_recent)
        frame[f"{side}_price_change_recently_observed"] = known_recent
        frame[f"{side}_price_change_age_censored_or_unknown"] = (
            price_valid & ~known_recent
        )

    for prefix, zone in (
        ("utc", "UTC"),
        ("london", "Europe/London"),
        ("new_york", "America/New_York"),
    ):
        local = frame["feature_timestamp_utc"].dt.tz_convert(zone)
        seconds = local.dt.hour * 3600 + local.dt.minute * 60 + local.dt.second
        angle = 2 * np.pi * seconds / 86400
        frame[f"{prefix}_hour"] = local.dt.hour
        frame[f"{prefix}_time_sin"] = np.sin(angle)
        frame[f"{prefix}_time_cos"] = np.cos(angle)

        if prefix != "utc":
            frame[f"{prefix}_session"] = (
                local.dt.dayofweek.lt(5)
                & local.dt.hour.ge(settings[f"{prefix}_session_start_hour"])
                & local.dt.hour.lt(settings[f"{prefix}_session_end_hour"])
            )

    frame["utc_weekday"] = frame["feature_timestamp_utc"].dt.dayofweek
    london, new_york = frame["london_session"], frame["new_york_session"]
    frame["state_reason"] = frame["state_reason"].astype("string")
    frame["session_overlap"] = london & new_york

    output = frame.loc[
        frame["feature_timestamp_utc"].ge(start)
        & frame["feature_timestamp_utc"].lt(end)
    ].reset_index(drop=True)

    expected_rows = int((end - start).total_seconds())
    if len(output) != expected_rows:
        raise RuntimeError("Clock-grid row conservation failed.")

    return normalise_feature_types(output)
'''.removeprefix("\\\n")

print("Sampling and feature calculations prepared.")

In [ ]:
# ## Storage and resource controls

# Processing uses bounded UTC blocks with preceding feature history.

# Each block loads:

# 1. The required trailing history.
# 2. The last source snapshot preceding that history.
# 3. Quotes through the block's final sampling timestamp.

# It does not load future label data.

# A completed block is committed before its checkpoint is updated.
# Reruns verify committed files and resume unfinished work.

# The new writer retries transient Windows replacement errors for a
# bounded period. It never deletes an existing checkpoint to force a
# replacement. The ingestion module remains unchanged.

# DuckDB filters the selected Parquet files before collecting a bounded
# # result into pandas.

In [ ]:
feature_storage = r'''

def _replace_path(temporary, target):
    delays = [0.1, 0.2, 0.4, 0.8, 1.6, 2.0, 2.0]
    for attempt in range(len(delays) + 1):
        try:
            os.replace(temporary, target)
            return
        except PermissionError as error:
            if (
                getattr(error, "winerror", None) not in {5, 32, 33}
                or attempt == len(delays)
            ):
                raise
            time.sleep(delays[attempt])


def write_json(path, value):
    path = Path(path)
    payload = json.dumps(
        value, indent=2, sort_keys=True, allow_nan=False
    ) + "\n"
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        descriptor, name = tempfile.mkstemp(
            prefix=f".{path.name}.", suffix=".tmp", dir=path.parent
        )
        temporary = Path(name)
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="\n") as handle:
            handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        _replace_path(temporary, path)
    finally:
        if temporary is not None:
            try:
                temporary.unlink(missing_ok=True)
            except OSError:
                pass


def _schema_hash(schema):
    return hashlib.sha256(
        schema.remove_metadata().serialize().to_pybytes()
    ).hexdigest()


def _save_frame(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    table = pa.Table.from_pandas(frame, preserve_index=False)
    descriptor, name = tempfile.mkstemp(
        prefix=f".{path.name}.", suffix=".tmp", dir=path.parent
    )
    temporary = Path(name)
    try:
        with os.fdopen(descriptor, "wb") as handle:
            pq.write_table(table, handle, compression="zstd")
            handle.flush()
            os.fsync(handle.fileno())
        _replace_path(temporary, path)
    finally:
        try:
            temporary.unlink(missing_ok=True)
        except OSError:
            pass

    return {
        "rows": len(frame),
        "bytes": path.stat().st_size,
        "sha256": _sha256(path),
        "schema_sha256": _schema_hash(table.schema),
    }


def _connect(files, view, working_directory, settings):
    temporary = Path(working_directory) / "duckdb_tmp"
    temporary.mkdir(parents=True, exist_ok=True)
    for name in ("duckdb_memory_limit", "duckdb_temp_limit"):
        if not re.fullmatch(r"\d+(MB|GB)", settings[name]):
            raise ValueError(f"Unsupported resource setting: {name}")

    connection = duckdb.connect(config={
        "memory_limit": settings["duckdb_memory_limit"],
        "threads": 2,
        "temp_directory": str(temporary),
    })
    try:
        # Set the spill cap after connecting; do not rely on connect(config=...).
        connection.execute(
            f"SET max_temp_directory_size = '{settings['duckdb_temp_limit']}'"
        )
        connection.execute("SET TimeZone = 'UTC'")
        connection.read_parquet(
            files, hive_partitioning=False
        ).create_view(view)
    except Exception:
        connection.close()
        raise
    return connection


def _source_files(source):
    root = Path(source["output_dir"])
    return [
        str(root / part["path"])
        for part in source["parts"] if part["kind"] == "quotes"
    ]


def _load_quotes(connection, start, end, settings):
    context = _utc(start) - pd.Timedelta(seconds=lookback_seconds(settings))
    last_grid = _utc(end) - pd.Timedelta(seconds=1)
    columns = ", ".join(QUOTE_COLUMNS)

    quotes = connection.execute(
        f"""
        SELECT {columns} FROM quotes
        WHERE timestamp_utc >= CAST(? AS TIMESTAMPTZ)
          AND timestamp_utc <= CAST(? AS TIMESTAMPTZ)
        ORDER BY timestamp_utc, source_row
        LIMIT ?
        """,
        [
            context.isoformat(), last_grid.isoformat(),
            settings["max_quotes_per_block"] + 1,
        ],
    ).fetchdf()

    if len(quotes) > settings["max_quotes_per_block"]:
        raise RuntimeError(
            "Quote block exceeded its row limit. Reduce block_hours "
            "before attempting a larger run."
        )

    predecessor = connection.execute(
        f"""
        SELECT {columns} FROM quotes
        WHERE timestamp_utc < CAST(? AS TIMESTAMPTZ)
        ORDER BY timestamp_utc DESC, source_row DESC
        LIMIT 1
        """,
        [context.isoformat()],
    ).fetchdf()

    return pd.concat([predecessor, quotes], ignore_index=True)


def _blocks(start, end, hours):
    if type(hours) is not int or not 1 <= hours <= 24:
        raise ValueError("block_hours must be an integer from 1 to 24.")
    left = _utc(start)
    end = _utc(end)
    output = []
    while left < end:
        right = min(
            left + pd.Timedelta(hours=hours),
            left.normalize() + pd.Timedelta(days=1),
            end,
        )
        output.append((left, right))
        left = right
    return output


def _frame_stats(frame, settings):
    return {
        "rows": len(frame),
        "price_valid_rows": int(frame["price_valid"].sum()),
        "size_pair_valid_rows": int(frame["size_pair_valid"].sum()),
        "wide_spread_observation_rows": int(
            frame["wide_spread_observation"].sum()
        ),
        "state_reasons": {
            str(key): int(value)
            for key, value in frame["state_reason"].value_counts().items()
        },
        "volatility_available_rows": {
            str(window): int(
                frame[f"realized_vol_{window}s_pips"].notna().sum()
            )
            for window in settings["volatility_windows_seconds"]
        },
    }


def _verify_feature_parts(state, expected_blocks):
    root = Path(state["output_dir"]).resolve()
    if len(state["parts"]) > len(expected_blocks):
        raise RuntimeError("Too many feature blocks in the checkpoint.")

    for part, (left, right) in zip(state["parts"], expected_blocks):
        if (part["start_utc"], part["end_utc"]) != (
            left.isoformat(), right.isoformat()
        ):
            raise RuntimeError("Feature checkpoint ranges are incompatible.")
        path = (root / part["path"]).resolve()
        if not path.is_relative_to(root):
            raise RuntimeError("Feature output escapes its generation directory.")
        if not path.is_file() or _sha256(path) != part["sha256"]:
            raise RuntimeError(f"Feature output checksum failed: {path}")
        parquet = pq.ParquetFile(path)
        if (
            parquet.metadata.num_rows != part["rows"]
            or part["rows"] != int((right - left).total_seconds())
            or _schema_hash(parquet.schema_arrow) != part["schema_sha256"]
        ):
            raise RuntimeError("Feature output schema or row count changed.")

    if state["status"] == "complete" and len(state["parts"]) != len(expected_blocks):
        raise RuntimeError("Completed feature generation has missing blocks.")
'''

print("Bounded storage helpers prepared.")

In [ ]:
# ## Feature generations

# The feature manifest identifies its exact ingestion input, feature
# configuration, implementation, package versions, and output blocks.

# Only manifest-listed files are read. Output directories are never
# globbed to combine potentially overlapping generations.

# The January generation will become the selected feature input for
# Notebook 03. The pilot remains a validation artifact.

# Memory diagnostics record the collected quote frame plus the returned
# feature frame. They are not measurements of total process peak memory.

In [ ]:
feature_pipeline = r'''

def build_feature_dataset(source_manifest_path, repository_root, start, end, settings):
    root = Path(repository_root).resolve()
    source_manifest_path = Path(source_manifest_path).resolve()
    source = json.loads(source_manifest_path.read_text(encoding="utf-8"))
    start, end = _utc(start), _utc(end)
    lookback_seconds(settings)

    if source["status"] != "complete" or not _source_files(source):
        raise ValueError("A completed ingestion generation is required.")
    if source.get("scope_flags", {}).get("flag_backward_timestamp", 0):
        raise ValueError("Resolve source-clock ordering reversals before features.")
    if not (
        pd.Timestamp(source["identity"]["start_utc"]) <= start
        < end <= pd.Timestamp(source["identity"]["end_utc"])
    ):
        raise ValueError("Feature range exceeds the selected ingestion range.")

    _verify_parts(source)
    for filename in _source_files(source):
        schema = pq.ParquetFile(filename).schema_arrow
        if any(name not in schema.names for name in QUOTE_COLUMNS):
            raise ValueError("Input Parquet lacks required quote columns.")
        field = schema.field("timestamp_utc")
        if not pa.types.is_timestamp(field.type) or field.type.tz != "UTC":
            raise ValueError("Input quote timestamps must explicitly carry UTC.")

    source_manifest_hash = _sha256(source_manifest_path)
    implementation = hashlib.sha256(
        Path(__file__).read_bytes()
        + Path(__file__).with_name("data.py").read_bytes()
    ).hexdigest()
    identity = {
        "input_manifest_sha256": source_manifest_hash,
        "input_job_id": source["job_id"],
        "start_utc": start.isoformat(),
        "end_utc": end.isoformat(),
        "settings": settings,
        "implementation_sha256": implementation,
        "versions": {
            name: version(name)
            for name in ("numpy", "pandas", "pyarrow", "duckdb", "pytz", "tzdata")
        },
    }
    job_id = hashlib.sha256(
        json.dumps(identity, sort_keys=True).encode("utf-8")
    ).hexdigest()[:24]
    output_dir = root / "data/processed/features" / job_id
    manifest_root = root / "artifacts/manifests/features"
    manifest_path = manifest_root / f"job-{job_id}.json"
    blocks = _blocks(start, end, settings["block_hours"])

    with _writer_lock(manifest_root):
        if manifest_path.exists():
            state = json.loads(manifest_path.read_text(encoding="utf-8"))
            if state["identity"] != identity:
                raise RuntimeError("Feature generation identity is incompatible.")
            _verify_feature_parts(state, blocks)
            if state["status"] == "complete":
                return {**state, "cache_hit": True}
        else:
            output_dir.mkdir(parents=True, exist_ok=True)
            state = {
                "kind": "feature_dataset",
                "job_id": job_id,
                "status": "running",
                "identity": identity,
                "input_manifest": str(source_manifest_path),
                "input_quote_rows": source["rows_in_scope"],
                "input_quote_utc_min": source["scope_utc_min"],
                "input_quote_utc_max": source["scope_utc_max"],
                "output_dir": str(output_dir),
                "manifest_path": str(manifest_path),
                "parts": [],
            }
            write_json(manifest_path, state)

        connection = _connect(
            _source_files(source), "quotes", output_dir, settings
        )
        try:
            for number, (left, right) in enumerate(blocks):
                if number < len(state["parts"]):
                    continue

                began = time.perf_counter()
                quotes = _load_quotes(connection, left, right, settings)
                frame = build_clock_features(
                    quotes, left, right, settings, source["scope_utc_min"]
                )

                future = (
                    frame["source_timestamp_utc"]
                    > frame["feature_timestamp_utc"]
                ).sum()
                if future:
                    raise RuntimeError("Future source observations entered features.")

                relative = (
                    f"year={left.year:04d}/month={left.month:02d}/"
                    f"part-{left.strftime('%Y%m%dT%H%M%S')}.parquet"
                )
                record = _save_frame(frame, output_dir / relative)
                if (
                    state["parts"]
                    and record["schema_sha256"]
                    != state["parts"][0]["schema_sha256"]
                ):
                    raise RuntimeError("Feature schema changed between blocks.")

                record.update({
                    "path": relative,
                    "start_utc": left.isoformat(),
                    "end_utc": right.isoformat(),
                    "statistics": _frame_stats(frame, settings),
                    "input_quotes_collected": len(quotes),
                    "observed_input_output_frame_bytes": int(
                        quotes.memory_usage(index=True, deep=True).sum()
                        + frame.memory_usage(index=True, deep=True).sum()
                    ),
                    "runtime_seconds": round(time.perf_counter() - began, 3),
                })
                state["parts"].append(record)
                write_json(manifest_path, state)

                if (number + 1) % 8 == 0 or number + 1 == len(blocks):
                    print(f"Feature blocks committed: {number + 1}/{len(blocks)}")
        finally:
            connection.close()

        if _sha256(source_manifest_path) != source_manifest_hash:
            raise RuntimeError("Input ingestion manifest changed during processing.")
        _verify_parts(source)
        _verify_feature_parts(state, blocks)
        state["status"] = "complete"
        write_json(manifest_path, state)
        return {**state, "cache_hit": False}


def load_feature_rows(state, start, end, settings, maximum_rows=100000):
    files = [
        str(Path(state["output_dir"]) / part["path"])
        for part in state["parts"]
    ]
    connection = _connect(files, "features", state["output_dir"], settings)
    try:
        frame = connection.execute("""
            SELECT * FROM features
            WHERE feature_timestamp_utc >= CAST(? AS TIMESTAMPTZ)
              AND feature_timestamp_utc < CAST(? AS TIMESTAMPTZ)
            ORDER BY feature_timestamp_utc
            LIMIT ?
        """, [
            _utc(start).isoformat(), _utc(end).isoformat(), maximum_rows + 1
        ]).fetchdf()
        if len(frame) > maximum_rows:
            raise RuntimeError("Requested pandas result exceeds its row limit.")
        return normalise_feature_types(frame)
    finally:
        connection.close()


def reference_feature_block(source, start, end, settings, working_directory):
    connection = _connect(
        _source_files(source), "quotes", working_directory, settings
    )
    try:
        quotes = _load_quotes(connection, start, end, settings)
        return build_clock_features(
            quotes, start, end, settings, source["scope_utc_min"]
        )
    finally:
        connection.close()


def summarise_features(state, settings):
    files = [
        str(Path(state["output_dir"]) / part["path"])
        for part in state["parts"]
    ]
    connection = _connect(files, "features", state["output_dir"], settings)
    try:
        rows, distinct_rows, minimum, maximum, future, stale_prices = (
            connection.execute("""
                SELECT count(*), count(DISTINCT feature_timestamp_utc),
                       min(feature_timestamp_utc), max(feature_timestamp_utc),
                       count(*) FILTER (
                           WHERE source_timestamp_utc > feature_timestamp_utc
                       ),
                       count(*) FILTER (
                           WHERE price_valid
                             AND snapshot_staleness_seconds > ?
                       )
                FROM features
            """, [settings["max_staleness_seconds"]]).fetchone()
        )
    finally:
        connection.close()

    expected = int((
        pd.Timestamp(state["identity"]["end_utc"])
        - pd.Timestamp(state["identity"]["start_utc"])
    ).total_seconds())

    if rows != expected or distinct_rows != rows or future or stale_prices:
        raise RuntimeError("Feature dataset validation failed.")

    totals = {
        "price_valid_rows": 0,
        "size_pair_valid_rows": 0,
        "wide_spread_observation_rows": 0,
    }
    reasons = {}
    volatility = {}
    for part in state["parts"]:
        statistics = part["statistics"]
        for key in totals:
            totals[key] += statistics[key]
        for key, value in statistics["state_reasons"].items():
            reasons[key] = reasons.get(key, 0) + value
        for key, value in statistics["volatility_available_rows"].items():
            volatility[key] = volatility.get(key, 0) + value

    return {
        "input_quote_rows": state["input_quote_rows"],
        "input_quote_utc_min": state["input_quote_utc_min"],
        "input_quote_utc_max": state["input_quote_utc_max"],
        "clock_rows": int(rows),
        "grid_utc_min": minimum.isoformat(),
        "grid_utc_max": maximum.isoformat(),
        **totals,
        "calendar_price_coverage_fraction": totals["price_valid_rows"] / rows,
        "state_reasons": reasons,
        "volatility_available_rows": volatility,
        "future_source_violations": int(future),
        "stale_prices_marked_usable": int(stale_prices),
        "parquet_bytes": sum(part["bytes"] for part in state["parts"]),
        "maximum_quotes_collected_per_block": max(
            part["input_quotes_collected"] for part in state["parts"]
        ),
        "maximum_observed_input_output_frame_bytes": max(
            part["observed_input_output_frame_bytes"] for part in state["parts"]
        ),
        "cache_hit": state["cache_hit"],
        "manifest": state["manifest_path"],
    }
'''

new_files["src/eurusd_research/features.py"] = (
    feature_calculations + feature_storage + feature_pipeline
)

print("Complete feature module prepared.")

In [ ]:
# ## Correctness tests

# The tests check:

# - Backward sampling and exact timestamp ties.
# - Pip and displayed-size calculations.
# - Invalid latest snapshots.
# - Staleness and source gaps.
# - Invalid observations within a second.
# - Zero size denominators.
# - Bounded observed-change ages.
# - Future-append causality.
# - Processing-boundary consistency.
# - London/New York daylight-saving behavior.
# - Checkpoint replacement retries.
# - Interrupted feature-generation recovery and damaged outputs.

# Tests construct small synthetic inputs in temporary directories.
# They do not alter historical data.

In [ ]:
new_files["tests/test_features.py"] = r'''\
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pytest

from eurusd_research import features as f


def settings(**changes):
    result = {
        "pip_size": 0.0001,
        "max_staleness_seconds": 5,
        "substantial_gap_seconds": 30,
        "return_windows_seconds": [1, 5],
        "volatility_windows_seconds": [5],
        "activity_windows_seconds": [1, 5],
        "change_windows_seconds": [1, 5],
        "observed_price_change_window_seconds": 10,
        "wide_spread_review_pips": 20,
        "london_session_start_hour": 8,
        "london_session_end_hour": 17,
        "new_york_session_start_hour": 8,
        "new_york_session_end_hour": 17,
        "block_hours": 6,
        "max_quotes_per_block": 10000,
        "max_grid_rows_per_block": 100000,
        "duckdb_memory_limit": "256MB",
        "duckdb_temp_limit": "4GB",
    }
    result.update(changes)
    return result


def quotes(times, bids=None, asks=None):
    times = pd.to_datetime(times, format="ISO8601", utc=True)
    n = len(times)
    bids = np.asarray(
        bids if bids is not None else 1.1 + np.arange(n) * 0.00001,
        dtype=float,
    )
    asks = np.asarray(
        asks if asks is not None else bids + 0.0002,
        dtype=float,
    )
    result = pd.DataFrame({
        "timestamp_utc": times,
        "source_row": np.arange(1, n + 1),
        "source_file": ["synthetic"] * n,
        "source_sha256": ["synthetic"] * n,
        "bid": bids,
        "ask": asks,
        "bid_size": np.full(n, 3.0),
        "ask_size": np.full(n, 1.0),
    })
    result["source_gap_seconds"] = result["timestamp_utc"].diff().dt.total_seconds()
    for name in (
        "flag_missing_size", "flag_invalid_size",
        "flag_zero_size", "flag_negative_size",
    ):
        result[name] = False
    return result


def test_backward_alignment_ties_and_units():
    q = quotes([
        "2020-02-01T00:00:00.200Z",
        "2020-02-01T00:00:01.200Z",
        "2020-02-01T00:00:02Z",
        "2020-02-01T00:00:02Z",
    ], [1.1, 1.1001, 1.1002, 1.1004])
    result = f.build_clock_features(
        q, "2020-02-01T00:00:01Z", "2020-02-01T00:00:03Z",
        settings(), q["timestamp_utc"].min(),
    )
    assert result.loc[0, "source_row"] == 1
    assert result.loc[1, "source_row"] == 4
    assert result.loc[0, "snapshot_staleness_seconds"] == pytest.approx(0.8)
    assert result.loc[0, "spread_pips"] == pytest.approx(2)
    assert result.loc[0, "size_imbalance"] == pytest.approx(0.5)
    assert result.loc[0, "size_weighted_minus_mid_pips"] == pytest.approx(0.5)
    assert result.loc[1, "snapshot_count_1s"] == 3


def test_invalid_latest_is_not_replaced_with_older_quote():
    q = quotes(
        ["2020-02-01T00:00:00Z", "2020-02-01T00:00:01Z"],
        [1.1, 1.101], [1.1002, 1.1005],
    )
    result = f.build_clock_features(
        q, "2020-02-01T00:00:01Z", "2020-02-01T00:00:02Z",
        settings(), q["timestamp_utc"].min(),
    )
    assert result.loc[0, "source_row"] == 2
    assert not result.loc[0, "price_valid"]
    assert pd.isna(result.loc[0, "mid_price"])


def test_stale_state_and_gap_break_returns():
    q = quotes(
        ["2020-02-01T00:00:00Z", "2020-02-01T00:00:31Z"]
    )
    result = f.build_clock_features(
        q, "2020-02-01T00:00:30Z", "2020-02-01T00:00:33Z",
        settings(max_staleness_seconds=30),
        q["timestamp_utc"].min(),
    )
    assert result.loc[0, "price_valid"]
    assert result.loc[1, "price_valid"]
    assert result.loc[1, "source_gap_observed_1s"]
    assert pd.isna(result.loc[1, "mid_change_1s_pips"])

    stale = f.build_clock_features(
        q, "2020-02-01T00:00:06Z", "2020-02-01T00:00:07Z",
        settings(), q["timestamp_utc"].min(),
    )
    assert stale.loc[0, "state_reason"] == "stale_snapshot"
    assert pd.isna(stale.loc[0, "bid"])


def test_intrasecond_invalid_quote_breaks_price_path():
    q = quotes([
        "2020-02-01T00:00:00Z",
        "2020-02-01T00:00:00.500Z",
        "2020-02-01T00:00:00.800Z",
    ], [1.1, 1.101, 1.1001], [1.1002, 1.1005, 1.1003])
    result = f.build_clock_features(
        q, "2020-02-01T00:00:01Z", "2020-02-01T00:00:02Z",
        settings(), q["timestamp_utc"].min(),
    )
    assert result.loc[0, "price_valid"]
    assert result.loc[0, "invalid_price_observed_1s"]
    assert result.loc[0, "price_path_break"]
    assert pd.isna(result.loc[0, "mid_change_1s_pips"])


def test_locked_prices_and_zero_denominator():
    q = quotes(["2020-02-01T00:00:00Z"], [1.1], [1.1])
    q[["bid_size", "ask_size"]] = 0.0
    q["flag_zero_size"] = True
    result = f.build_clock_features(
        q, "2020-02-01T00:00:00Z", "2020-02-01T00:00:01Z",
        settings(), q["timestamp_utc"].min(),
    )
    assert result.loc[0, "price_valid"]
    assert result.loc[0, "locked_quote"]
    assert not result.loc[0, "size_pair_valid"]
    assert pd.isna(result.loc[0, "size_imbalance"])


def test_change_age_at_and_beyond_cap():
    q = quotes([
        "2020-02-01T00:00:00Z",
        "2020-02-01T00:00:01Z",
        "2020-02-01T00:00:11Z",
        "2020-02-01T00:00:12Z",
    ], [1.1, 1.1001, 1.1001, 1.1001])
    result = f.build_clock_features(
        q, "2020-02-01T00:00:11Z", "2020-02-01T00:00:13Z",
        settings(), q["timestamp_utc"].min(),
    )
    assert result.loc[0, "bid_price_change_age_seconds"] == 10
    assert pd.isna(result.loc[1, "bid_price_change_age_seconds"])
    assert result.loc[1, "bid_price_change_age_censored_or_unknown"]


def test_future_append_and_partition_consistency():
    times = pd.date_range("2020-01-31T23:55:00Z", periods=1000, freq="1s")
    q = quotes(times)
    cfg = settings()
    start, boundary, end = (
        "2020-01-31T23:58:00Z",
        "2020-02-01T00:00:00Z",
        "2020-02-01T00:06:00Z",
    )
    coverage = q["timestamp_utc"].min()
    full = f.build_clock_features(q, start, end, cfg, coverage)
    pieces = pd.concat([
        f.build_clock_features(q, start, boundary, cfg, coverage),
        f.build_clock_features(q, boundary, end, cfg, coverage),
    ], ignore_index=True)
    pd.testing.assert_frame_equal(full, pieces, rtol=1e-10, atol=1e-12)

    cutoff = pd.Timestamp(boundary)
    earlier_input = q.loc[q["timestamp_utc"].lt(cutoff)]
    before = f.build_clock_features(
        earlier_input, start, boundary, cfg, coverage
    )
    after = f.build_clock_features(q, start, boundary, cfg, coverage)
    pd.testing.assert_frame_equal(before, after)


@pytest.mark.parametrize("timestamp, london_hour, new_york_hour", [
    ("2025-03-03T12:00:00Z", 12, 7),
    ("2025-03-17T12:00:00Z", 12, 8),
    ("2025-03-31T12:00:00Z", 13, 8),
])
def test_local_time_features(timestamp, london_hour, new_york_hour):
    q = quotes([timestamp])
    end = (pd.Timestamp(timestamp) + pd.Timedelta(seconds=1)).isoformat()
    result = f.build_clock_features(
        q, timestamp, end, settings(), q["timestamp_utc"].min()
    )
    assert result.loc[0, "london_hour"] == london_hour
    assert result.loc[0, "new_york_hour"] == new_york_hour


def test_checkpoint_retries_and_preserves_previous_value(tmp_path, monkeypatch):
    path = tmp_path / "checkpoint.json"
    f.write_json(path, {"version": 1})
    actual_replace = f.os.replace
    calls = 0

    def transient_lock(source, target):
        nonlocal calls
        calls += 1
        if calls == 1:
            error = PermissionError("simulated sharing error")
            error.winerror = 32
            raise error
        return actual_replace(source, target)

    monkeypatch.setattr(f.os, "replace", transient_lock)
    monkeypatch.setattr(f.time, "sleep", lambda seconds: None)
    f.write_json(path, {"version": 2})
    assert json.loads(path.read_text()) == {"version": 2}
    assert calls == 2

    def persistent_lock(source, target):
        error = PermissionError("simulated persistent sharing error")
        error.winerror = 32
        raise error

    monkeypatch.setattr(f.os, "replace", persistent_lock)
    with pytest.raises(PermissionError):
        f.write_json(path, {"version": 3})
    assert json.loads(path.read_text()) == {"version": 2}


def test_feature_job_resume_and_corruption(tmp_path, monkeypatch):
    q = quotes(pd.date_range(
        "2020-01-31T23:55:00Z", periods=1000, freq="1s"
    ))
    input_dir = tmp_path / "input"
    input_dir.mkdir()
    parquet = input_dir / "quotes.parquet"
    pq.write_table(pa.Table.from_pandas(q, preserve_index=False), parquet)
    source = {
        "job_id": "synthetic-source",
        "status": "complete",
        "output_dir": str(input_dir),
        "identity": {
            "start_utc": "2020-01-31T00:00:00Z",
            "end_utc": "2020-02-02T00:00:00Z",
        },
        "rows_in_scope": len(q),
        "rows_unassigned": 0,
        "scope_utc_min": q["timestamp_utc"].min().isoformat(),
        "scope_utc_max": q["timestamp_utc"].max().isoformat(),
        "scope_flags": {"flag_backward_timestamp": 0},
        "parts": [{
            "path": parquet.name,
            "kind": "quotes",
            "rows": len(q),
            "bytes": parquet.stat().st_size,
            "sha256": f._sha256(parquet),
        }],
    }
    manifest = input_dir / "manifest.json"
    f.write_json(manifest, source)
    cfg = settings()
    start, end = "2020-01-31T23:58:00Z", "2020-02-01T00:06:00Z"

    original_write = f.write_json
    interrupted = False

    def save_then_interrupt(path, value):
        nonlocal interrupted
        original_write(path, value)
        if (
            value.get("kind") == "feature_dataset"
            and value["status"] == "running"
            and len(value["parts"]) == 1
            and not interrupted
        ):
            interrupted = True
            raise RuntimeError("simulated interruption")

    monkeypatch.setattr(f, "write_json", save_then_interrupt)
    with pytest.raises(RuntimeError, match="simulated interruption"):
        f.build_feature_dataset(manifest, tmp_path / "run", start, end, cfg)

    resumed = f.build_feature_dataset(
        manifest, tmp_path / "run", start, end, cfg
    )
    reference = f.reference_feature_block(
        source, start, end, cfg, tmp_path / "reference"
    )
    stored = f.load_feature_rows(resumed, start, end, cfg)
    pd.testing.assert_frame_equal(
        reference, stored, check_dtype=False, rtol=1e-10, atol=1e-12
    )

    repeated = f.build_feature_dataset(
        manifest, tmp_path / "run", start, end, cfg
    )
    assert repeated["cache_hit"]

    part = Path(resumed["output_dir"]) / resumed["parts"][0]["path"]
    part.write_bytes(b"damaged")
    with pytest.raises(RuntimeError, match="checksum"):
        f.build_feature_dataset(manifest, tmp_path / "run", start, end, cfg)
'''.removeprefix("\\\n")

print("Feature correctness tests prepared.")

In [ ]:
# ## Create files and extend configuration

# The next cell creates the two new files only if absent, or accepts
# identical existing contents. Conflicting files stop execution.

# Shared settings are added without replacing existing values.
# Machine-specific paths remain in ignored local configuration.

# No additional package installation is required.

In [ ]:
example_path = REPO_ROOT / "configs/research_config.example.json"
example = json.loads(example_path.read_text(encoding="utf-8-sig"))

sampling_defaults = {
    "pip_size": 0.0001,
    "max_staleness_seconds": 5,
    "substantial_gap_seconds": 30,
    "return_windows_seconds": [1, 5, 30, 60],
    "volatility_windows_seconds": [60, 300],
    "activity_windows_seconds": [1, 5, 30, 60, 300],
    "change_windows_seconds": [1, 5],
    "observed_price_change_window_seconds": 300,
    "wide_spread_review_pips": 20,
    "london_session_start_hour": 8,
    "london_session_end_hour": 17,
    "new_york_session_start_hour": 8,
    "new_york_session_end_hour": 17,
    "block_hours": 6,
    "max_quotes_per_block": 750000,
    "max_grid_rows_per_block": 100000,
    "duckdb_memory_limit": "256MB",
    "duckdb_temp_limit": "4GB",
}

sampling = example.setdefault("sampling_features", {})
for key, value in sampling_defaults.items():
    sampling.setdefault(key, value)

review_defaults = {
    "spread_pips_threshold": 20,
    "continuous_duration_seconds_threshold": 1200,
    "comparison": "strictly_greater",
    "requires_fresh_quote_coverage": True,
    "action": "investigate",
    "implementation_status": "pending_liquidity_episode_analysis",
}
review = example.setdefault("sustained_wide_spread_review", {})
for key, value in review_defaults.items():
    review.setdefault(key, value)

for relative, text in new_files.items():
    path = REPO_ROOT / relative
    if not path.resolve().is_relative_to(REPO_ROOT):
        raise ValueError("Target escapes the repository.")
    if path.exists() and (
        not path.is_file()
        or path.read_text(encoding="utf-8-sig") != text
    ):
        raise FileExistsError(
            f"Existing contents differ: {relative}. Nothing was written."
        )

for relative, text in new_files.items():
    path = REPO_ROOT / relative
    if path.exists():
        print("MATCHES", relative)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("x", encoding="utf-8", newline="\n") as handle:
            handle.write(text)
        print("CREATED", relative)

example_path.write_text(
    json.dumps(example, indent=2) + "\n",
    encoding="utf-8",
    newline="\n",
)

print("Shared configuration extended.")

In [ ]:
# ## Verify imports and select the input

# The package must import from the notebooks directory without
# modifying Python's module search path.

# The input is the selected January ingestion manifest. The separate
# pilot-day ingestion output is not added.

In [ ]:
probe = subprocess.run(
    [
        sys.executable, "-c",
        (
            "import json, eurusd_research; "
            "from eurusd_research import features; "
            "print(json.dumps({'package_file': eurusd_research.__file__, "
            "'feature_file': features.__file__}))"
        ),
    ],
    cwd=REPO_ROOT / "notebooks",
    capture_output=True,
    text=True,
    check=True,
)

import_probe = json.loads(probe.stdout)
expected = REPO_ROOT / "src/eurusd_research/features.py"
if Path(import_probe["feature_file"]).resolve() != expected.resolve():
    raise RuntimeError("A different feature-module installation was imported.")

importlib.invalidate_caches()

from eurusd_research.config import load_config
from eurusd_research.features import (
    build_clock_features,
    build_feature_dataset,
    load_feature_rows,
    normalise_feature_types,
    reference_feature_block,
    summarise_features,
    write_json,
)

config = load_config(REPO_ROOT)
settings = config["sampling_features"]

if settings["pip_size"] != config["pip_size"]:
    raise ValueError("Conflicting pip-size settings.")

SOURCE_MANIFEST = REPO_ROOT / config["active_ingestion_manifest"]
source_state = json.loads(SOURCE_MANIFEST.read_text(encoding="utf-8"))

print("Input ingestion job:", source_state["job_id"])
print("Input quote rows:", f"{source_state['rows_in_scope']:,}")
print("Input UTC coverage:", source_state["scope_utc_min"], "to",
      source_state["scope_utc_max"])
print(json.dumps(import_probe, indent=2))

In [ ]:
tests_passed = False

test_path = REPO_ROOT / "tests" / "test_features.py"
old = "times = pd.to_datetime(times, utc=True)"
new = 'times = pd.to_datetime(times, format="ISO8601", utc=True)'

contents = test_path.read_text(encoding="utf-8")

if contents.count(old) == 1:
    test_path.write_text(contents.replace(old, new), encoding="utf-8")
    print("Corrected the synthetic timestamp parser.")
elif old not in contents and new in contents:
    print("Synthetic timestamp parser is already corrected.")
else:
    raise RuntimeError(
        "Unexpected test-file contents. Stop and return this output."
    )

test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)

if test_run.returncode != 0:
    raise RuntimeError("Tests failed. Stop and return Cell 16 output.")

tests_passed = True
test_summary = test_run.stdout.strip().splitlines()[-1]

In [ ]:
import numpy as np
import pandas as pd

from eurusd_research.config import load_config
from eurusd_research.features import build_clock_features

demo = pd.DataFrame({
    "timestamp_utc": pd.to_datetime(
        [
            "2020-02-01T00:00:00.200Z",
            "2020-02-01T00:00:01.200Z",
            "2020-02-01T00:00:02Z",
            "2020-02-01T00:00:02Z",
        ],
        format="ISO8601",
        utc=True,
    ),
    "source_row": np.arange(1, 5, dtype=np.int64),
    "source_file": "synthetic-demo",
    "source_sha256": "synthetic-demo",
    "bid": [1.1000, 1.1001, 1.1002, 1.1004],
    "ask": [1.1002, 1.1003, 1.1004, 1.1006],
    "bid_size": 3.0,
    "ask_size": 1.0,
})

demo["source_gap_seconds"] = (
    demo["timestamp_utc"].diff().dt.total_seconds()
)

for flag in [
    "flag_missing_size",
    "flag_invalid_size",
    "flag_zero_size",
    "flag_negative_size",
]:
    demo[flag] = False

demo_features = build_clock_features(
    demo,
    start="2020-02-01T00:00:00Z",
    end="2020-02-01T00:00:04Z",
    settings=load_config(REPO_ROOT)["sampling_features"],
    coverage_start=demo["timestamp_utc"].iloc[0],
)

assert len(demo_features) == 4
assert pd.isna(demo_features.iloc[0]["mid_price"])
assert demo_features["source_row"].iloc[1:].tolist() == [1, 4, 4]

assert np.isclose(
    demo_features.iloc[1]["snapshot_staleness_seconds"], 0.8
)

for column, expected in [
    ("spread_pips", 2.0),
    ("size_imbalance", 0.5),
    ("size_weighted_minus_mid_pips", 0.5),
]:
    np.testing.assert_allclose(
        demo_features[column].iloc[1:],
        expected,
        atol=1e-10,
    )

print(demo_features[[
    "feature_timestamp_utc",
    "source_timestamp_utc",
    "source_row",
    "mid_price",
    "spread_pips",
    "size_imbalance",
    "snapshot_staleness_seconds",
]].to_string(index=False))

print(
    "\nPASS: backward alignment, timestamp ties, staleness, "
    "pip units, imbalance, and size-weighted price offset."
)

In [ ]:
import shutil
import time

assert tests_passed

if shutil.disk_usage(REPO_ROOT).free < 6 * 1024**3:
    raise RuntimeError("Free at least 6 GiB before processing.")

day_start = config["ingestion"]["day_start_utc"]
day_end = config["ingestion"]["day_end_utc"]
measurements = {}

began = time.perf_counter()
day_features_state = build_feature_dataset(
    SOURCE_MANIFEST, REPO_ROOT, day_start, day_end, settings
)
day_feature_summary = summarise_features(day_features_state, settings)
measurements["day_execution_seconds"] = round(time.perf_counter() - began, 3)

continuous_day = reference_feature_block(
    source_state,
    day_start,
    day_end,
    settings,
    REPO_ROOT / "reports/private/notebook02_reference",
)
stored_day = load_feature_rows(
    day_features_state, day_start, day_end, settings, maximum_rows=100000
)

pd.testing.assert_frame_equal(
    continuous_day,
    stored_day,
    check_dtype=False,
    rtol=1e-10,
    atol=1e-12,
)
day_partition_check = "passed"

print("Continuous day versus stored blocks: PASS")
print(json.dumps(day_feature_summary, indent=2))

del continuous_day, stored_day

In [ ]:
RUN_MONTH_FEATURES = True

if not RUN_MONTH_FEATURES:
    raise RuntimeError("Month feature generation deliberately postponed.")

if shutil.disk_usage(REPO_ROOT).free < 6 * 1024**3:
    raise RuntimeError("Free at least 6 GiB before the month run.")

month_start = source_state["identity"]["start_utc"]
month_end = source_state["identity"]["end_utc"]

began = time.perf_counter()
month_features_state = build_feature_dataset(
    SOURCE_MANIFEST, REPO_ROOT, month_start, month_end, settings
)
month_feature_summary = summarise_features(month_features_state, settings)
measurements["month_execution_seconds"] = round(time.perf_counter() - began, 3)

print(json.dumps(month_feature_summary, indent=2))

In [ ]:
assert tests_passed
assert day_partition_check == "passed"

local_path = REPO_ROOT / "configs/research_config.local.json"
local = json.loads(local_path.read_text(encoding="utf-8-sig"))
local["active_features_manifest"] = str(
    Path(month_features_state["manifest_path"]).relative_to(REPO_ROOT)
).replace("\\", "/")
write_json(local_path, local)

private = REPO_ROOT / "reports/private"
private.mkdir(parents=True, exist_ok=True)

environment_versions = {
    name: version(name)
    for name in (
        "numpy", "pandas", "pyarrow", "duckdb", "pytz", "pytest", "tzdata"
    )
}
write_json(private / "notebook02_execution.json", {
    "settings": settings,
    "day": day_feature_summary,
    "month": month_feature_summary,
    "measurements": measurements,
    "versions": environment_versions,
    "pytest": test_summary,
    "day_partition_check": day_partition_check,
    "owner_feature_review": "pending",
})

write_json(
    REPO_ROOT / "artifacts/manifests/reviews/notebook01_checkpoint.json",
    {
        "checkpoint": "Select the January generation to avoid pilot duplication.",
        "owner_answer": (
            "We should use the January dataset to avoid duplication of data."
        ),
        "confirmed": True,
        "scope": "Dataset selection and overlapping-generation duplication",
        "checkpoint_error_workaround": "Owner paused OneDrive synchronization.",
        "checkpoint_code_change_reported": False,
    },
)


def marked_update(path, tag, content):
    existing = path.read_text(encoding="utf-8-sig") if path.exists() else ""
    beginning, ending = f"<!-- BEGIN {tag} -->", f"<!-- END {tag} -->"
    block = beginning + "\n" + content.rstrip() + "\n" + ending

    if beginning in existing or ending in existing:
        if existing.count(beginning) != 1 or existing.count(ending) != 1:
            raise ValueError(f"Invalid generated markers in {path.name}")
        start = existing.index(beginning)
        finish = existing.index(ending, start) + len(ending)
        text = existing[:start] + block + existing[finish:]
    else:
        text = existing.rstrip() + "\n\n" + block + "\n"

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")


dictionary = f"""\
# Feature definitions

All feature timestamps and source timestamps are UTC.

- Mid, bid, ask, and size-weighted quote price: USD per EUR.
- Spread and weighted quote displacement: pips; one pip is {settings['pip_size']}.
- Displayed sizes and size changes: original provider units, unresolved.
- Size imbalance and log returns: dimensionless.
- Snapshot staleness and observed price-change ages: seconds.
- Snapshot counts: source records, not trades.

Return windows: {settings['return_windows_seconds']} seconds.
Volatility windows: {settings['volatility_windows_seconds']} seconds.
Activity windows: {settings['activity_windows_seconds']} seconds.
Spread/size-change windows: {settings['change_windows_seconds']} seconds.

Price changes compare t with t-h. Realized volatility sums squared
one-second pip changes over (t-W, t], then takes the square root.
All intervening price steps must be usable.

Snapshot counts include source records in (t-W, t], including different
updates sharing a timestamp. Insufficient initial history remains missing.

Observed bid/ask price-change ages are available only within
{settings['observed_price_change_window_seconds']} seconds. Invalid
comparisons and substantial source gaps reset knowledge. These are not
true side-specific quote ages.

London and New York sessions use local Monday–Friday hours
08:00 inclusive to 17:00 exclusive. These are research conventions.

The size-weighted quote price is a candidate feature.
No forecasting benefit or calibrated fair value has been established.

A spread above 20 pips for more than 1200 continuous seconds is the
owner's proposed investigation criterion. Its episode analysis remains
pending; it is not an exclusion rule here.
"""
(REPO_ROOT / "reports/feature_dictionary.md").write_text(
    dictionary, encoding="utf-8", newline="\n"
)

marked_update(
    REPO_ROOT / "reports/data_quality.md",
    "NOTEBOOK02",
    f"""\
## Notebook 02 — sampling coverage

- Calendar grid rows: {month_feature_summary['clock_rows']:,}
- Usable-price rows: {month_feature_summary['price_valid_rows']:,}
- Valid displayed-size-pair rows:
  {month_feature_summary['size_pair_valid_rows']:,}
- Future-source violations:
  {month_feature_summary['future_source_violations']}
- Stale prices marked usable:
  {month_feature_summary['stale_prices_marked_usable']}
- Continuous-day versus block comparison: {day_partition_check}

Coverage includes closed periods and time before the first source quote.
It does not measure feed uptime.

Missing feature values are preserved. No model imputation or forecast
evaluation has occurred. Wide spreads are not automatically excluded.
""",
)

marked_update(
    REPO_ROOT / "README.md",
    "NOTEBOOK02",
    """\
## Implemented sampling and features

Notebook 02 creates a one-second UTC grid from manifest-selected quotes.
Sampling is backward-looking, with explicit staleness and continuity
checks. Prices become unavailable when the latest state is unusable.

Features include prices, spreads, displayed sizes, imbalance,
size-weighted quote displacement, trailing price changes and log returns,
realized volatility, snapshot activity, spread/size changes, time features,
and bounded observed price-change ages.

See reports/feature_dictionary.md for units and window definitions.
The active month feature manifest is recorded in ignored local configuration.

Tests cover financial units, temporal alignment, future-append causality,
processing boundaries, DST-aware local times, and checkpoint recovery.

Results pending execution and validation.
""",
)

marked_update(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK01_CHECKPOINT",
    """\
## Confirmed Notebook 01 checkpoint

Raymond Fendi confirmed that January research should use the January
dataset to avoid duplicating observations from the overlapping pilot day.

He reported resolving the checkpoint replacement error by pausing
OneDrive synchronization, without reporting a checkpoint-code change.

He proposed investigating spreads above 20 pips when they persist for
more than 20 continuous minutes. This is a domain-directed investigation
criterion; measured episode analysis remains pending.
""",
)

marked_update(
    REPO_ROOT / "CONTRIBUTIONS.md",
    "NOTEBOOK02",
    """\
## Notebook 02 implementation

OpenAI Codex assisted with causal sampling, feature implementation,
correctness tests, bounded processing, and documentation.

This section is generated after the notebook's tests, pilot comparison,
and month feature generation complete.

Raymond Fendi's review of the new feature mechanisms remains pending.
No predictive or profitability result is claimed.
""",
)

print("Active feature manifest:", local["active_features_manifest"])
print("Execution diagnostics: reports/private/notebook02_execution.json")
print("Feature dictionary: reports/feature_dictionary.md")

In [ ]:
def compact_feature_summary(value):
    result = dict(value)
    result["manifest"] = str(
        Path(result["manifest"]).relative_to(REPO_ROOT)
    ).replace("\\", "/")
    return result


summary = {
    "notebook": "02_sampling_and_features.ipynb",
    "python_version": sys.version.split()[0],
    "package_versions": environment_versions,
    "input_ingestion_job": source_state["job_id"],
    "configuration": settings,
    "validation": {
        "imports_from_notebooks_directory": "passed",
        "pytest": test_summary,
        "synthetic_alignment_checks": "passed",
        "day_continuous_vs_blocks": day_partition_check,
        "future_source_violations": (
            month_feature_summary["future_source_violations"]
        ),
        "stale_prices_marked_usable": (
            month_feature_summary["stale_prices_marked_usable"]
        ),
        "owner_notebook01_dataset_checkpoint": "confirmed",
        "owner_notebook02_review": "pending",
    },
    "day": compact_feature_summary(day_feature_summary),
    "month": compact_feature_summary(month_feature_summary),
    "measurements": measurements,
    "warnings": [
        "Calendar coverage includes closed periods and initial unavailable history.",
        "The staleness limit is an initial research assumption.",
        "Displayed-size denomination remains unresolved.",
        "Observed price-change ages are bounded and are not true quote ages.",
        "Frame memory measurements are not total process peak memory.",
        "Sustained-wide-spread episode analysis remains pending.",
        "No labels, predictive evaluation, or profitability findings exist yet.",
    ],
    "saved_outputs": {
        "active_features_manifest": local["active_features_manifest"],
        "feature_dictionary": "reports/feature_dictionary.md",
        "execution_diagnostics": "reports/private/notebook02_execution.json",
        "data_quality_report": "reports/data_quality.md",
    },
}

print("NOTEBOOK_02_SUMMARY")
print(json.dumps(summary, indent=2, allow_nan=False))

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from eurusd_research.config import load_config
from eurusd_research.features import load_feature_rows

coverage_started = time.perf_counter()
coverage_config = load_config(REPO_ROOT)

coverage_manifest = Path(coverage_config["active_features_manifest"])
if not coverage_manifest.is_absolute():
    coverage_manifest = REPO_ROOT / coverage_manifest

coverage_state = json.loads(
    coverage_manifest.read_text(encoding="utf-8")
)
if coverage_state["status"] != "complete":
    raise RuntimeError("A completed feature generation is required.")

# Use the settings that actually produced these features.
coverage_settings = coverage_state["identity"]["settings"]
coverage_start = pd.Timestamp(coverage_state["identity"]["start_utc"])
coverage_end = pd.Timestamp(coverage_state["identity"]["end_utc"])

lookbacks = (60, 300)
intervals = (1, 5, 10)
available = {(window, interval): 0
             for window in lookbacks for interval in intervals}

usable_total = 0
clock_total = 0

for left in pd.date_range(
    coverage_start, coverage_end, freq="6h", inclusive="left"
):
    right = min(left + pd.Timedelta(hours=6), coverage_end)
    history_start = max(
        coverage_start,
        left - pd.Timedelta(seconds=max(lookbacks)),
    )

    frame = load_feature_rows(
        coverage_state,
        history_start,
        right,
        coverage_settings,
        maximum_rows=22000,
    )

    timestamps = frame["feature_timestamp_utc"]
    expected_rows = int((right - history_start).total_seconds())

    assert len(frame) == expected_rows
    assert timestamps.iloc[0] == history_start
    assert timestamps.diff().dropna().eq(
        pd.Timedelta(seconds=1)
    ).all()

    # Count only this block; preceding rows supply rolling history.
    current = timestamps.ge(left).to_numpy()
    usable = frame["price_valid"].to_numpy(dtype=bool)
    barriers = (
        frame["source_gap_observed_1s"]
        | frame["invalid_price_observed_1s"]
    )

    clock_total += int(current.sum())
    usable_total += int(np.count_nonzero(current & usable))

    for window in lookbacks:
        # Reject known invalid events or substantial gaps in (t-window, t].
        clean_window = barriers.rolling(
            window, min_periods=window
        ).sum().eq(0).to_numpy()

        for interval in intervals:
            eligible = clean_window.copy()

            if interval == 1:
                eligible &= pd.Series(usable).rolling(
                    window + 1, min_periods=window + 1
                ).sum().eq(window + 1).to_numpy()
            else:
                # Require usable prices at t, t-interval, ..., t-window.
                for lag in range(0, window + 1, interval):
                    if lag == 0:
                        eligible &= usable
                    else:
                        eligible[lag:] &= usable[:-lag]
                        eligible[:lag] = False

            if interval == 1:
                reference = frame[
                    f"realized_vol_{window}s_pips"
                ].notna().to_numpy()

                if not np.array_equal(
                    eligible[current], reference[current]
                ):
                    raise AssertionError(
                        f"{window}s baseline reconstruction differs. "
                        "Stop and return this output."
                    )

            available[(window, interval)] += int(
                np.count_nonzero(eligible & current)
            )

    del frame

expected_usable = sum(
    part["statistics"]["price_valid_rows"]
    for part in coverage_state["parts"]
)
assert usable_total == expected_usable
assert clock_total == int((coverage_end - coverage_start).total_seconds())
assert usable_total > 0

for window in lookbacks:
    assert (
        available[(window, 1)]
        <= available[(window, 5)]
        <= available[(window, 10)]
    )

coverage_results = pd.DataFrame([
    {
        "lookback_seconds": window,
        "return_interval_seconds": interval,
        "returns_per_window": window // interval,
        "required_prices": window // interval + 1,
        "available_rows": available[(window, interval)],
        "usable_price_coverage_pct": round(
            100 * available[(window, interval)] / usable_total, 2
        ),
        "reaches_80_percent": (
            available[(window, interval)] / usable_total >= 0.80
        ),
    }
    for window in lookbacks
    for interval in intervals
])

print("VOLATILITY_COVERAGE_CHECK")
print(f"Period: {coverage_start} to {coverage_end} (end exclusive)")
print(f"Usable-price denominator: {usable_total:,}")
print(
    "Maximum endpoint staleness:",
    coverage_settings["max_staleness_seconds"],
    "seconds",
)
print()
print(coverage_results.to_string(index=False))
print("\nPASS: existing baselines reproduced and coverage counts reconciled.")
print(f"Runtime: {time.perf_counter() - coverage_started:.2f} seconds")

In [1]:
import json
from pathlib import Path

import duckdb
import pandas as pd
from IPython.display import display

from eurusd_research.config import load_config
from eurusd_research.features import load_feature_rows

# Find the project without assuming where Jupyter was started.
current_directory = Path.cwd().resolve()
root = next(
    (
        path
        for path in (current_directory, *current_directory.parents)
        if (path / ".git").exists()
    ),
    None,
)
if root is None:
    root = Path(input("Repository location: ")).expanduser().resolve()

def project_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else root / path

config = load_config(root)
feature_state = json.loads(
    project_path(config["active_features_manifest"]).read_text(
        encoding="utf-8"
    )
)
ingestion_state = json.loads(
    project_path(feature_state["input_manifest"]).read_text(
        encoding="utf-8"
    )
)

assert feature_state["status"] == "complete"
assert ingestion_state["status"] == "complete"

settings = feature_state["identity"]["settings"]
staleness_limit = settings["max_staleness_seconds"]

quote_files = [
    str(Path(ingestion_state["output_dir"]) / part["path"])
    for part in ingestion_state["parts"]
    if part["kind"] == "quotes"
]

with duckdb.connect(
    config={
        "memory_limit": settings["duckdb_memory_limit"],
        "threads": 2,
    }
) as connection:
    connection.execute("SET TimeZone = 'UTC'")
    connection.read_parquet(
        quote_files, hive_partitioning=False
    ).create_view("quotes")

    # Choose a short intraday pause, rather than a weekend closure.
    candidates = connection.execute("""
        SELECT timestamp_utc, source_row, source_file,
               source_sha256, source_gap_seconds
        FROM quotes
        WHERE source_gap_seconds > ?
          AND source_gap_seconds <= ?
          AND EXTRACT(
              ISODOW FROM timezone('Europe/London', timestamp_utc)
          ) BETWEEN 1 AND 5
          AND EXTRACT(
              HOUR FROM timezone('Europe/London', timestamp_utc)
          ) >= ?
          AND EXTRACT(
              HOUR FROM timezone('Europe/London', timestamp_utc)
          ) < ?
        ORDER BY abs(source_gap_seconds - 10.0),
                 timestamp_utc, source_row
        LIMIT 1
    """, [
        staleness_limit + 2,
        settings["substantial_gap_seconds"],
        settings["london_session_start_hour"],
        settings["london_session_end_hour"],
    ]).fetchdf()

    if candidates.empty:
        raise RuntimeError(
            "No qualifying intraday gap was found. Return this message."
        )

    selected = candidates.iloc[0]
    gap_end_row = int(selected["source_row"])

    tick_excerpt = connection.execute("""
        SELECT timestamp_utc, source_row,
               bid, ask, bid_size, ask_size, source_gap_seconds
        FROM quotes
        WHERE source_sha256 = ?
          AND source_row BETWEEN ? AND ?
        ORDER BY source_row
    """, [
        str(selected["source_sha256"]),
        max(1, gap_end_row - 6),
        gap_end_row + 6,
    ]).fetchdf()

before = tick_excerpt.loc[
    tick_excerpt["source_row"].eq(gap_end_row - 1), "timestamp_utc"
]
after = tick_excerpt.loc[
    tick_excerpt["source_row"].eq(gap_end_row), "timestamp_utc"
]

if len(before) != 1 or len(after) != 1:
    raise RuntimeError("Could not identify both gap boundaries.")

before_timestamp = before.iloc[0]
after_timestamp = after.iloc[0]
observed_gap = (after_timestamp - before_timestamp).total_seconds()

assert abs(observed_gap - float(selected["source_gap_seconds"])) < 1e-6

tick_excerpt["boundary"] = ""
tick_excerpt.loc[
    tick_excerpt["source_row"].eq(gap_end_row - 1), "boundary"
] = "BEFORE GAP"
tick_excerpt.loc[
    tick_excerpt["source_row"].eq(gap_end_row), "boundary"
] = "AFTER GAP"

clock_excerpt = load_feature_rows(
    feature_state,
    before_timestamp.floor("s") - pd.Timedelta(seconds=1),
    after_timestamp.ceil("s") + pd.Timedelta(seconds=2),
    settings,
    maximum_rows=100,
)

inside_gap = (
    clock_excerpt["feature_timestamp_utc"].gt(before_timestamp)
    & clock_excerpt["feature_timestamp_utc"].lt(after_timestamp)
)
stale_seconds = int(
    (
        inside_gap
        & clock_excerpt["state_reason"].eq("stale_snapshot")
    ).sum()
)

print("Source file:", Path(str(selected["source_file"])).name)
print(f"Observed interval between ticks: {observed_gap:.3f} seconds")
print(f"Maximum accepted snapshot age: {staleness_limit} seconds")
print(f"Stale sampled seconds inside this interval: {stale_seconds}")

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.precision", 6,
):
    print("\nACTUAL TICKS — original source ordering")
    display(tick_excerpt)

    print("\nSTORED ONE-SECOND STATES — timestamps are UTC")
    display(clock_excerpt[[
        "feature_timestamp_utc",
        "source_timestamp_utc",
        "snapshot_staleness_seconds",
        "price_valid",
        "state_reason",
        "mid_price",
        "price_path_break",
        "realized_vol_60s_pips",
    ]])

Source file: EURUSD_Ticks_2016.01.04_2017.01.02.csv
Observed interval between ticks: 10.001 seconds
Maximum accepted snapshot age: 5 seconds
Stale sampled seconds inside this interval: 5

ACTUAL TICKS — original source ordering


,timestamp_utc,source_row,bid,ask,bid_size,ask_size,source_gap_seconds,boundary
0,2016-01-18 15:17:01.591000+00:00,1083197,1.08908,1.08910,2.25,1.50,0.254,
1,2016-01-18 15:17:02.097000+00:00,1083198,1.08909,1.08911,1.00,6.90,0.506,
2,2016-01-18 15:17:02.619000+00:00,1083199,1.08909,1.08910,1.00,1.00,0.522,
3,2016-01-18 15:17:02.721000+00:00,1083200,1.08907,1.08909,1.50,2.25,0.102,
4,2016-01-18 15:17:02.823000+00:00,1083201,1.08906,1.08907,1.87,1.42,0.102,
5,2016-01-18 15:17:03.679000+00:00,1083202,1.08904,1.08907,1.87,1.50,0.856,BEFORE GAP
6,2016-01-18 15:17:13.680000+00:00,1083203,1.08903,1.08906,3.75,2.62,10.001,AFTER GAP
7,2016-01-18 15:17:13.731000+00:00,1083204,1.08903,1.08906,3.07,2.62,0.051,
8,2016-01-18 15:17:16.273000+00:00,1083205,1.08903,1.08905,1.57,1.00,2.542,
9,2016-01-18 15:17:16.528000+00:00,1083206,1.08901,1.08903,1.00,1.00,0.255,



STORED ONE-SECOND STATES — timestamps are UTC


,feature_timestamp_utc,source_timestamp_utc,snapshot_staleness_seconds,price_valid,state_reason,mid_price,price_path_break,realized_vol_60s_pips
0,2016-01-18 15:17:02+00:00,2016-01-18 15:17:01.591000+00:00,0.409,True,usable_price,1.089090,False,NaN
1,2016-01-18 15:17:03+00:00,2016-01-18 15:17:02.823000+00:00,0.177,True,usable_price,1.089065,False,NaN
2,2016-01-18 15:17:04+00:00,2016-01-18 15:17:03.679000+00:00,0.321,True,usable_price,1.089055,False,NaN
3,2016-01-18 15:17:05+00:00,2016-01-18 15:17:03.679000+00:00,1.321,True,usable_price,1.089055,False,NaN
4,2016-01-18 15:17:06+00:00,2016-01-18 15:17:03.679000+00:00,2.321,True,usable_price,1.089055,False,NaN
5,2016-01-18 15:17:07+00:00,2016-01-18 15:17:03.679000+00:00,3.321,True,usable_price,1.089055,False,NaN
6,2016-01-18 15:17:08+00:00,2016-01-18 15:17:03.679000+00:00,4.321,True,usable_price,1.089055,False,NaN
7,2016-01-18 15:17:09+00:00,2016-01-18 15:17:03.679000+00:00,5.321,False,stale_snapshot,NaN,True,NaN
8,2016-01-18 15:17:10+00:00,2016-01-18 15:17:03.679000+00:00,6.321,False,stale_snapshot,NaN,True,NaN
9,2016-01-18 15:17:11+00:00,2016-01-18 15:17:03.679000+00:00,7.321,False,stale_snapshot,NaN,True,NaN


In [2]:
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

from eurusd_research.config import load_config
from eurusd_research.features import (
    build_clock_features,
    build_feature_dataset,
    summarise_features,
    write_json,
)

current_directory = Path.cwd().resolve()
policy_root = next(
    (
        path
        for path in (current_directory, *current_directory.parents)
        if (path / ".git").exists()
    ),
    None,
)
if policy_root is None:
    policy_root = Path(
        input("Repository location: ")
    ).expanduser().resolve()

def policy_path(value):
    path = Path(value).expanduser()
    return (path if path.is_absolute() else policy_root / path).resolve()

policy_config = load_config(policy_root)
policy_local_path = policy_root / "configs/research_config.local.json"
policy_local = json.loads(
    policy_local_path.read_text(encoding="utf-8-sig")
)

# Preserve the original comparison reference on subsequent reruns.
reference_key = "notebook02_5s_reference_manifest"
policy_baseline_path = policy_path(
    policy_local.get(
        reference_key,
        policy_config["active_features_manifest"],
    )
)
policy_baseline = json.loads(
    policy_baseline_path.read_text(encoding="utf-8")
)

assert policy_baseline["status"] == "complete"
assert (
    policy_baseline["identity"]["settings"]["max_staleness_seconds"] == 5
), "The comparison reference must be the original five-second generation."

policy_settings = dict(policy_config["sampling_features"])
policy_settings.update(
    max_staleness_seconds=30,
    substantial_gap_seconds=30,
)

# One snapshot: ages 29 and 30 must be valid; age 31 must be stale.
boundary_quotes = pd.DataFrame({
    "timestamp_utc": pd.to_datetime(
        ["2020-02-01T00:00:00Z"], format="ISO8601", utc=True
    ),
    "source_row": [1],
    "source_file": ["synthetic"],
    "source_sha256": ["synthetic"],
    "bid": [1.1000],
    "ask": [1.1002],
    "bid_size": [3.0],
    "ask_size": [1.0],
    "source_gap_seconds": [float("nan")],
})
for flag in (
    "flag_missing_size", "flag_invalid_size",
    "flag_zero_size", "flag_negative_size",
):
    boundary_quotes[flag] = False

boundary_result = build_clock_features(
    boundary_quotes,
    "2020-02-01T00:00:29Z",
    "2020-02-01T00:00:32Z",
    policy_settings,
    boundary_quotes["timestamp_utc"].iloc[0],
)

assert boundary_result["price_valid"].tolist() == [True, True, False]
assert boundary_result["snapshot_staleness_seconds"].tolist() == [
    29.0, 30.0, 31.0
]

policy_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--color=no"],
    cwd=policy_root,
    capture_output=True,
    text=True,
)
print(policy_tests.stdout)
if policy_tests.stderr:
    print(policy_tests.stderr)
if policy_tests.returncode != 0:
    raise RuntimeError("Tests failed. Stop and return this cell's output.")

policy_test_summary = policy_tests.stdout.strip().splitlines()[-1]
print("PASS: snapshot ages 29s/30s accepted; 31s rejected.")
print("Policy prepared. Configuration has not yet been changed.")

.......................                                                  [100%]
23 passed in 3.35s

PASS: snapshot ages 29s/30s accepted; 31s rejected.
Policy prepared. Configuration has not yet been changed.


In [4]:
if shutil.disk_usage(policy_root).free < 6 * 1024**3:
    raise RuntimeError("Free at least 6 GiB before regenerating features.")

policy_started = time.perf_counter()

# A manifest loaded from disk has no execution-specific cache status.
policy_baseline_summary_input = policy_baseline.copy()
policy_baseline_summary_input["cache_hit"] = None

policy_old_summary = summarise_features(
    policy_baseline_summary_input,
    policy_baseline["identity"]["settings"],
)

policy_new_state = build_feature_dataset(
    policy_path(policy_baseline["input_manifest"]),
    policy_root,
    policy_baseline["identity"]["start_utc"],
    policy_baseline["identity"]["end_utc"],
    policy_settings,
)
policy_new_summary = summarise_features(
    policy_new_state, policy_settings
)

assert (
    policy_new_state["identity"]["input_job_id"]
    == policy_baseline["identity"]["input_job_id"]
)
assert (
    policy_new_summary["clock_rows"]
    == policy_old_summary["clock_rows"]
)
assert (
    policy_new_summary["price_valid_rows"]
    >= policy_old_summary["price_valid_rows"]
)
assert policy_new_summary["future_source_violations"] == 0
assert policy_new_summary["stale_prices_marked_usable"] == 0

policy_new_manifest = Path(
    policy_new_state["manifest_path"]
).resolve().relative_to(policy_root).as_posix()

policy_reference_manifest = (
    policy_baseline_path.relative_to(policy_root).as_posix()
)

# Set the shared research policy while preserving other configuration.
policy_example_path = (
    policy_root / "configs/research_config.example.json"
)
policy_example = json.loads(
    policy_example_path.read_text(encoding="utf-8-sig")
)
policy_example["sampling_features"].update(
    max_staleness_seconds=30,
    substantial_gap_seconds=30,
)

# Remove conflicting local overrides; retain paths and other settings.
local_sampling = policy_local.get("sampling_features", {})
local_sampling.pop("max_staleness_seconds", None)
local_sampling.pop("substantial_gap_seconds", None)

policy_local[reference_key] = policy_reference_manifest
policy_local["active_features_manifest"] = policy_new_manifest

write_json(policy_example_path, policy_example)
write_json(policy_local_path, policy_local)

effective_policy = load_config(policy_root)["sampling_features"]
assert effective_policy == policy_settings

policy_comparison = {
    "notebook": "02_sampling_and_features.ipynb",
    "processing_scope": {
        "start_utc": policy_new_state["identity"]["start_utc"],
        "end_utc_exclusive": policy_new_state["identity"]["end_utc"],
        "input_quote_rows": policy_new_summary["input_quote_rows"],
        "clock_rows": policy_new_summary["clock_rows"],
    },
    "policy": {
        "maximum_snapshot_age_seconds": 30,
        "substantial_gap_seconds": 30,
        "full_history_processing": "pending; shared policy now configured",
        "ten_second_volatility_implementation": "pending",
    },
    "comparison": {
        "price_valid_rows_at_5s": policy_old_summary["price_valid_rows"],
        "price_valid_rows_at_30s": policy_new_summary["price_valid_rows"],
        "additional_price_valid_rows": (
            policy_new_summary["price_valid_rows"]
            - policy_old_summary["price_valid_rows"]
        ),
        "size_pair_valid_rows_at_30s": (
            policy_new_summary["size_pair_valid_rows"]
        ),
        "calendar_price_coverage_pct_at_30s": round(
            100 * policy_new_summary["calendar_price_coverage_fraction"],
            2,
        ),
        "volatility_available_rows_at_30s": (
            policy_new_summary["volatility_available_rows"]
        ),
        "state_reasons_at_30s": policy_new_summary["state_reasons"],
    },
    "validation": {
        "pytest": policy_test_summary,
        "30s_boundary_check": "passed",
        "future_source_violations": 0,
        "snapshots_older_than_30s_marked_usable": 0,
    },
    "runtime_seconds": round(time.perf_counter() - policy_started, 3),
    "parquet_bytes": policy_new_summary["parquet_bytes"],
    "cache_hit": policy_new_summary["cache_hit"],
    "saved_outputs": {
        "baseline_manifest": policy_reference_manifest,
        "active_manifest": policy_new_manifest,
        "diagnostics": "reports/private/staleness_30s_update.json",
    },
}

write_json(
    policy_root / "reports/private/staleness_30s_update.json",
    policy_comparison,
)

# Replace only this amendment's section when the cell is rerun.
def record_policy_note(relative_path, body):
    path = policy_root / relative_path
    text = path.read_text(encoding="utf-8")
    begin = "<!-- BEGIN STALENESS_30S -->"
    end = "<!-- END STALENESS_30S -->"
    block = f"{begin}\n{body.strip()}\n{end}"

    if begin in text:
        if text.count(begin) != 1 or text.count(end) != 1:
            raise RuntimeError(f"Malformed policy markers: {relative_path}")
        before, remainder = text.split(begin, 1)
        _, after = remainder.split(end, 1)
        text = before + block + after
    else:
        if end in text:
            raise RuntimeError(f"Malformed policy markers: {relative_path}")
        text = text.rstrip() + "\n\n" + block + "\n"

    path.write_text(text, encoding="utf-8", newline="\n")

policy_note = f"""
## Current snapshot eligibility policy

Raymond Fendi selected a maximum accepted snapshot age of 30 seconds.
Ages up to and including 30 seconds are accepted when prices are valid.
Source gaps exceeding 30 seconds still break return continuity.

January was regenerated from the same ingestion generation:
{policy_old_summary['price_valid_rows']:,} usable-price rows under the
five-second rule versus {policy_new_summary['price_valid_rows']:,}
under the thirty-second rule. The original generation is retained.

The earlier Notebook 02 coverage figures describe the five-second baseline.
The shared configuration now specifies thirty seconds for future processing.
Full-history regeneration remains pending.

Automated validation: {policy_test_summary}; the exact thirty-second
boundary check passed. This records execution evidence, not a new owner
code-review confirmation.

The selected ten-second volatility implementation remains pending.
Results pending execution and validation.
"""

for document in (
    "README.md",
    "CONTRIBUTIONS.md",
    "reports/data_quality.md",
    "reports/feature_dictionary.md",
):
    record_policy_note(document, policy_note)

print("STALENESS_30S_SUMMARY")
print(json.dumps(policy_comparison, indent=2))

Feature blocks committed: 8/124
Feature blocks committed: 16/124
Feature blocks committed: 24/124
Feature blocks committed: 32/124
Feature blocks committed: 40/124
Feature blocks committed: 48/124
Feature blocks committed: 56/124
Feature blocks committed: 64/124
Feature blocks committed: 72/124
Feature blocks committed: 80/124
Feature blocks committed: 88/124
Feature blocks committed: 96/124
Feature blocks committed: 104/124
Feature blocks committed: 112/124
Feature blocks committed: 120/124
Feature blocks committed: 124/124
STALENESS_30S_SUMMARY
{
  "notebook": "02_sampling_and_features.ipynb",
  "processing_scope": {
    "start_utc": "2016-01-01T00:00:00+00:00",
    "end_utc_exclusive": "2016-02-01T00:00:00+00:00",
    "input_quote_rows": 2163940,
    "clock_rows": 2678400
  },
  "policy": {
    "maximum_snapshot_age_seconds": 30,
    "substantial_gap_seconds": 30,
    "full_history_processing": "pending; shared policy now configured",
    "ten_second_volatility_implementation": "pen